# 🎙️ HooperTTS + Qwen3-TTS + Phi-3.5-mini

An open-source narration compiler for expressive AI voice generation.

**Model roles in this Colab:**

- **Script enhancement:** `microsoft/Phi-3.5-mini-instruct`
- **Voice cloning / TTS:** `Qwen/Qwen3-TTS-12Hz-1.7B-Base`

Supported

• Voice Cloning  
• Gaming News  
• Documentary  
• YouTube Shorts  
• Podcast

GitHub:
https://github.com/XADITYAM/HooperTTS


## First Run

The first execution downloads the Qwen3-TTS model (roughly 4.5 GB) and may also cache
Phi-3.5-mini-instruct for script enhancement.

HooperTTS loads the **Phi enhancement model first**, validates/retries the rewrite,
then releases Phi before loading Qwen3-TTS. This keeps the two models from occupying
GPU memory at the same time.


In [ ]:
import os

# Always pull the latest code from GitHub when the repo already exists too.
# If you are testing an unpushed local ZIP release (for example HooperTTS-main-fixed-phi-v3),
# replace this cell with an upload/unzip step instead of pulling GitHub.
if not os.path.exists('/content/HooperTTS'):
    !git clone https://github.com/XADITYAM/HooperTTS.git
else:
    !cd /content/HooperTTS && git pull

%cd /content/HooperTTS


In [ ]:
# Install HooperTTS plus the optional [enhancement] extra and Gradio requirements.
# The enhancement extra provides Transformers + Accelerate, which Phi uses.
!pip install -e '.[enhancement]'
!pip install -r requirements.txt


In [ ]:
%cd /content

if not os.path.exists('/content/Qwen3-TTS'):
    !git clone https://github.com/QwenLM/Qwen3-TTS.git

%cd /content/Qwen3-TTS


In [ ]:
!pip install -e .

# HooperTTS's Qwen runner writes WAV output with soundfile.
!pip install soundfile


In [ ]:
# --- Permanent fix for the Xet/CAS 403 SignatureError ---
# Must run BEFORE imports that may initialize huggingface_hub.
import os

!pip uninstall -y hf-xet -q
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '30'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'

print('Xet disabled, plain HTTP downloads forced.')


In [ ]:
import torch

print('=' * 48)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'Free VRAM: {free_bytes / 1024**3:.1f} GiB / {total_bytes / 1024**3:.1f} GiB')
    print('Phi-3.5-mini enhancement target: ~6 GiB free VRAM')
    print('Qwen3-TTS target: ~4.5 GiB model footprint')
    print('HooperTTS loads them sequentially, not concurrently.')
print('PyTorch:', torch.__version__)
print('=' * 48)


In [ ]:
# Optional pre-cache for the Phi enhancement model.
# This verifies the Colab runtime can reach the model before you press Generate.
# The actual model weights are still loaded lazily by HooperTTS when enhancement runs.
import time
from huggingface_hub import snapshot_download


def robust_snapshot_download(repo_id, max_retries=5, backoff_seconds=10, **kwargs):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id=repo_id, **kwargs)
        except Exception as exc:
            last_err = exc
            wait = backoff_seconds * attempt
            print(f'[attempt {attempt}/{max_retries}] download failed: {exc}')
            if attempt < max_retries:
                print(f'Retrying in {wait}s...')
                time.sleep(wait)
    raise last_err

phi_path = robust_snapshot_download(
    repo_id='microsoft/Phi-3.5-mini-instruct',
    max_workers=1,
)
print('Phi cache:', phi_path)


In [ ]:
# Pre-cache the Qwen3-TTS checkpoint with the same retry logic.
import time
from huggingface_hub import snapshot_download


def robust_snapshot_download(repo_id, max_retries=5, backoff_seconds=10, **kwargs):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id=repo_id, **kwargs)
        except Exception as exc:
            last_err = exc
            wait = backoff_seconds * attempt
            print(f'[attempt {attempt}/{max_retries}] download failed: {exc}')
            if attempt < max_retries:
                print(f'Retrying in {wait}s...')
                time.sleep(wait)
    raise last_err

model_path = robust_snapshot_download(
    repo_id='Qwen/Qwen3-TTS-12Hz-1.7B-Base',
    max_workers=1,
)
print('Qwen3-TTS cache:', model_path)


In [ ]:
%cd /content/HooperTTS
!python -m pytest -q
!hoopertts doctor


## Script Enhancement — Phi-3.5-mini-instruct

The current HooperTTS enhancement pipeline is configured around **Phi-3.5-mini-instruct** for the Creative tier.

Use:

- **Enhance script** — run Phi and return the validated rewrite.
- **Enhance + optimize** — run Phi, validate it, then run HooperTTS narration optimization.
- **Off (optimize only)** — skip the LLM and use the existing optimizer.

The enhancement backend now:

1. Builds an immutable-fact ledger from the source (dates, years, numbered titles, URLs/prices and other high-risk spans).
2. Gives Phi explicit instructions to preserve those facts while rewriting structure and delivery.
3. Strictly validates the candidate after generation.
4. Retries once by default when protected facts are missing, feeding the exact validator failures back to Phi.
5. Falls back to the original script if the retry still fails.

For your GTA-style scripts, facts such as `GTA 6`, `Grand Theft Auto 6`, `Aug. 27`, `2022`, and `Red Dead Redemption 2` must survive enhancement unchanged in substance.

**Important:** Phi is the *script* model. Qwen3-TTS remains the *voice synthesis / cloning* model.


In [ ]:
# Make the Colab app open with Phi selected by default.
# The app still exposes Qwen3-1.7B and Qwen3-0.6B as alternatives.
# This is a runtime-only patch to the cloned checkout, so the GitHub repository is not modified.
from pathlib import Path

app_file = Path('/content/HooperTTS/app.py')
text = app_file.read_text(encoding='utf-8')
old = 'DEFAULT_ENHANCEMENT_MODEL_LABEL = "Quality — Qwen3-1.7B (recommended)"'
new = 'DEFAULT_ENHANCEMENT_MODEL_LABEL = "Creative — Phi-3.5-mini (bolder rewrites, ~6 GiB VRAM)"'

if old in text:
    app_file.write_text(text.replace(old, new, 1), encoding='utf-8')
    print('✓ Colab default changed to Phi-3.5-mini.')
elif new in text:
    print('✓ Phi-3.5-mini is already the app default.')
else:
    raise RuntimeError('Could not find the enhancement model default in app.py; refusing to patch blindly.')


In [ ]:
%cd /content/HooperTTS

# Launch HooperTTS.
# In the UI, use 'Enhance + optimize (recommended)' with
# 'Creative — Phi-3.5-mini' for the full Phi -> validator/retry -> optimizer -> Qwen3-TTS path.
!python app.py
